# 🎙️ speakerscribe — lote de transcripción + diarización (Drive + Colab)

Procesa **todos** los audios/videos de `data/` y deja un `.txt` por grabación con el **mismo nombre del audio** (incluido el `*`), con hablantes y tiempos. Es reanudable: si Colab corta la sesión, al volver a ejecutar retoma exactamente donde quedó, sin repetir lo confirmado.

**Uso normal:** *Entorno de ejecución → Ejecutar todo*. Solo se edita la **celda 3**.

| Carpeta bajo la raíz | Contenido |
|---|---|
| `data/` | 📥 Sus audios y videos (subcarpetas incluidas). Lo transcrito OK sale de aquí. |
| `entregables/` | ⭐ `<nombre>.txt` por grabación + `_resumen.md` + `.speakerscribe_state/` (**no borrar**) |
| `transcripts/` | Formatos opcionales: `.transcript.md`, `.srt`, `.json`, `.plano.txt` |
| `splits/` | Texto para LLM: `.full_for_llm.txt` (y partes, si las activa) |
| `_procesados/AAAA-MM-DD/` | Audios ya transcritos; se borran solos a los 30 días |

**Requisitos:** GPU T4 (*Entorno de ejecución → Cambiar tipo → T4 GPU*) y el secreto `HF_TOKEN` (🔑, token *Read* con las condiciones aceptadas en [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1)).

**Leyenda:** 🟢 obligatoria · 🟡 opcional

## 🟢 1 · Montar Google Drive

In [ ]:
# 🟢 CELDA 1 — Drive (entradas y resultados duraderos; los temporales van al disco local)
from google.colab import drive

drive.mount('/content/drive')

## 🟢 2 · Instalar speakerscribe (a lo sumo **un** reinicio planificado)

Instala la versión fijada en `VERSION`. `ORIGEN='auto'` usa PyPI y, si la versión aún no está publicada, la instala directo desde GitHub (`main`). `'wheel'` usa un `.whl` de `RUTA_WHEELS`.

**Por qué puede pedir un reinicio:** pip puede actualizar binarios ya cargados en la sesión (`numpy`, `torch`). La celda lo detecta y reinicia **una vez**; lo instalado persiste. Al volver, ejecute de nuevo las celdas 1 y 2 (la 2 termina en segundos).

In [ ]:
# 🟢 CELDA 2 — instalación verificada
VERSION = '0.4.0'            # ⇦ versión de la librería que usa este notebook
ORIGEN = 'auto'              # 'auto' (PyPI → GitHub) · 'pypi' · 'github' · 'wheel'
RUTA_WHEELS = '/content/drive/MyDrive/Pruebas/Speakerscribe/dist'   # solo para ORIGEN='wheel'
RUTA_CODIGO = '/content/drive/MyDrive/Pruebas/Speakerscribe'        # copia de desarrollo en Drive
REINICIO_AUTOMATICO = True

import importlib
import importlib.metadata as im
import os
import subprocess
import sys
import time
from pathlib import Path

GITHUB = 'git+https://github.com/EnriqueForero/speakerscribe@main'
CRITICOS = ('numpy', 'torch', 'torchaudio', 'torchvision')
codigo = Path(RUTA_CODIGO)


def _ver(pkg):
    try:
        return im.version(pkg)
    except im.PackageNotFoundError:
        return None


def _dentro(p):
    try:
        rp = Path(p or '.').resolve()
    except OSError:
        return False
    return rp == codigo or codigo in rp.parents


def _pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout[-4000:] + r.stderr[-4000:])


def _verificar():
    import speakerscribe
    origen = Path(speakerscribe.__file__).resolve()
    if _dentro(str(origen)):
        raise RuntimeError(f'El import resolvió a la copia de Drive ({origen}). Reinicie y ejecute desde la celda 1.')
    if speakerscribe.__version__ != VERSION:
        raise RuntimeError(f'Se importó v{speakerscribe.__version__}, se esperaba v{VERSION}. Reinicie el entorno.')
    av = _ver('av') or '0'
    if int(av.split('.')[0]) >= 19:
        raise RuntimeError(f'av {av} es incompatible con faster-whisper 1.2.1 (requiere av<19).')
    return speakerscribe.__version__


# Higiene: una carpeta speakerscribe/ en el CWD o en sys.path ensombrece al paquete instalado.
sys.path[:] = [p for p in sys.path if not (p and _dentro(p))]
if _dentro(os.getcwd()):
    os.chdir('/content')

t0 = time.time()
if _ver('speakerscribe') == VERSION:
    print(f'⚡ speakerscribe v{_verificar()} ya instalado y verificado ({time.time() - t0:.1f} s).')
else:
    antes = {p: _ver(p) for p in CRITICOS}
    ya_importado = 'speakerscribe' in sys.modules
    candidatos = []
    if ORIGEN == 'wheel':
        ruedas = sorted(Path(RUTA_WHEELS).glob(f'speakerscribe-{VERSION}-*.whl'))
        if not ruedas:
            raise FileNotFoundError(f'No hay speakerscribe-{VERSION}-*.whl en {RUTA_WHEELS}')
        candidatos = [str(ruedas[-1])]
    if ORIGEN in ('auto', 'pypi'):
        candidatos.append(f'speakerscribe=={VERSION}')
    if ORIGEN in ('auto', 'github'):
        candidatos.append(f'speakerscribe @ {GITHUB}')
    for spec in candidatos:
        print(f'Instalando {spec} (~2-3 min)…')
        ok, salida = _pip(spec)
        if ok:
            break
        print(f'   ↳ no disponible por esta vía; se intenta la siguiente.')
    else:
        print(salida)
        raise RuntimeError('pip no pudo instalar speakerscribe por ninguna vía (detalle arriba).')
    importlib.invalidate_caches()
    deriva = [f'{p}: {antes[p]} → {_ver(p)}' for p in CRITICOS
              if antes[p] and _ver(p) and antes[p] != _ver(p)]
    if deriva or ya_importado:
        print('pip actualizó binarios ya cargados:', *deriva, sep='\n   ♻️ ')
        print('\n══ REINICIO PLANIFICADO (1 de 1): al volver, ejecute las celdas 1 y 2 ══')
        if REINICIO_AUTOMATICO:
            sys.stdout.flush()
            time.sleep(5)
            os.kill(os.getpid(), 9)
        raise RuntimeError('Reinicie el entorno y ejecute de nuevo las celdas 1 y 2.')
    print(f'✔ speakerscribe v{_verificar()} listo en {time.time() - t0:.0f} s.')

## 🟢 3 · ✏️ Configuración — la única celda que se edita

Todas las rutas salen de `RAIZ`. Cada valor se valida al construir `settings`: un error de tipeo falla **aquí**, no a mitad del lote. Cambiar un formato o un nombre de hablante **no** gasta GPU (se re-renderiza desde el JSON maestro); cambiar modelo, idioma, beam, hablantes o glosario sí (ver `POLITICA_CAMBIO_PERFIL`).

In [ ]:
# 🟢 CELDA 3 — CONFIGURACIÓN (rutas, motor, entregables, retención, apagado)
from speakerscribe.batch import BatchPaths, BatchSettings

RAIZ = '/content/drive/MyDrive/ProColombia/1B. Resultados/Transcripcion-Diarizacion'  # ⇦ raíz del proyecto

settings = BatchSettings(
    root=RAIZ,
    # input_dir=None, deliverables_dir=None, formats_dir=None, llm_dir=None,   # ⇦ None = <RAIZ>/data, /entregables, /transcripts, /splits

    # ── Motor (cambiarlo implica GPU) ─────────────────────────────────
    model='large-v3',            # 'large-v3' (máxima calidad) · 'large-v3-turbo' · 'medium' · 'small'
    language='es',               # '' = autodetección por archivo
    beam_size=5,                 # medido en T4: beam 5 ⇒ RTF ~21x
    batch_size=8,                # 8 = rápido en T4 · 1 = ruta secuencial bit-exacta
    anti_hallucination=True,     # corta bucles de repetición en audios largos
    glossary='',                 # nombres propios/jerga (lo importante AL FINAL); por archivo: <nombre>.prompt.txt
    num_speakers=0,              # 0 = desconocido; si sabe el número EXACTO, mejora la precisión
    min_speakers=1,
    max_speakers=20,

    # ── Entregables (cambiarlos NO gasta GPU) ──────────────────────────
    deliver_full_llm=True,       # splits/<nombre>.full_for_llm.txt (texto corrido para un LLM)
    deliver_markdown=False,      # transcripts/<nombre>.transcript.md
    deliver_srt=False,           # transcripts/<nombre>.srt
    deliver_json=False,          # transcripts/<nombre>.json
    deliver_plain=False,         # transcripts/<nombre>.plano.txt
    deliver_splits=False,        # splits/<nombre>.parte_NN.txt (~split_words palabras)
    include_header=True,
    timestamp_ms=False,
    name_template='{stem}',      # {stem} = nombre del audio TAL CUAL (conserva el *)

    # ── Calidad y reintentos ───────────────────────────────────────────
    max_attempts=2,
    profile_change_policy='keep_and_report',   # 'keep_and_report' · 'reprocess' · 'stop'
    on_output_changed='report',  # .txt que usted movió/renombró/editó: 'report' lo respeta
    force_reprocess=False,       # ⚠️ True rehace TODO (ignora confirmados y caché de diarización)

    # ── Retención ──────────────────────────────────────────────────────
    after_success='move_to_processed',   # el audio OK pasa a _procesados/AAAA-MM-DD/ · 'keep' lo deja en data/
    processed_retention_days=30,         # días en _procesados antes de borrarse
    diar_cache_retention_days=90,        # caché de diarización: 3 meses

    # ── Sesión, recursos y apagado ─────────────────────────────────────
    max_session_minutes=650,     # cierre ordenado antes del tope de Colab (0 = sin límite)
    max_files_per_session=0,     # 0 = sin límite (1 = corrida de prueba)
    ram_pct_recycle=70,
    ram_pct_stop=88,
    shutdown_at_end=True,        # libera la GPU al terminar (cancelable con ⏹ durante 180 s)
)

rutas = BatchPaths.from_settings(settings)
print('✔ Configuración válida')
for etiqueta, ruta in (('Entrada', rutas.input), ('Entregables', rutas.deliverables),
                       ('Formatos', rutas.formats), ('Texto LLM', rutas.llm),
                       ('Procesados', rutas.processed), ('Estado', rutas.state),
                       ('Temporales (local)', rutas.scratch)):
    print(f'  {etiqueta:<20} {ruta}')

## 🟡 4 · Revincular la entrada (una sola vez tras mover `data/`)

El estado recuerda la carpeta de entrada para no mezclar colecciones. Si movió `data/` a propósito (p. ej. de `Pruebas/Speakerscribe/data` a la ruta nueva), la celda 6 se detendrá pidiendo esto. Ponga `REVINCULAR = True`, ejecute **una vez** y vuelva a `False`. Queda registrado en el journal y se guarda respaldo del vínculo anterior.

In [ ]:
# 🟡 CELDA 4 — revincular (solo una vez, cuando la celda 6 lo pida)
from speakerscribe.batch import rebind_workspace

REVINCULAR = False  # ⇦ True, ejecutar una vez, volver a False

if REVINCULAR:
    r = rebind_workspace(settings)
    print('✔ Revinculado:', (r['anterior'] or {}).get('input_root'), '→', r['nuevo']['input_root'])
    print('  Respaldo:', r['respaldo'])
else:
    print('⏩ Sin cambios (REVINCULAR=False).')

## 🟡 5 · Estado global (sin GPU, solo lectura)

In [ ]:
# 🟡 CELDA 5 — censo: confirmados, pendientes, marcados, inválidos
from speakerscribe.batch import status

print('\n'.join(status(settings).lines()))

## 🟢 6 · 🚀 Ejecutar el lote

Por cada archivo: verificar → transcribir + diarizar → publicar → JSON maestro → commit → mover el audio a `_procesados/`. Todo queda confirmado en Drive **antes** de pasar al siguiente. Protecciones: guardia de RAM, presupuesto de sesión, corte ante fallos de entorno repetidos (no gasta intentos) y apagado solo si hubo trabajo sin errores.

In [ ]:
%%time
# 🟢 CELDA 6 — EJECUCIÓN (reanudable; el apagado depende de la configuración)
from speakerscribe.batch import run_batch

REPORTE = run_batch(settings)

## 🟡 7 · Renombrar hablantes (sin GPU)

Con `RUN_RENAME = False` lista lo publicado. Copie el nombre del `.txt` o del audio en `ARCHIVO`, edite `MAPPING` y ponga `RUN_RENAME = True`. Todos los entregables de esa grabación se regeneran con los nombres reales y los conservan en el futuro.

In [ ]:
# 🟡 CELDA 7 — renombrar hablantes y re-renderizar (CPU)
from speakerscribe.batch import published, rename_speakers

ARCHIVO = ''   # ⇦ nombre del .txt o del audio, tal como aparece en la lista
MAPPING = {'SPEAKER_00': 'Entrevistador', 'SPEAKER_01': 'Invitada'}
RUN_RENAME = False

if not RUN_RENAME:
    for fila in published(settings, limit=25):
        print(f"  · {fila['salida']}   (audio: {fila['audio']})")
    print('\n⏩ Edite ARCHIVO y MAPPING y ponga RUN_RENAME=True.')
else:
    registro = rename_speakers(settings, ARCHIVO, MAPPING)
    print('✔ Re-renderizado:', registro['output']['relative_path'], '·', len(registro.get('extras') or []), 'extra(s)')

## 🟢 8 · 🔬 Autopsia — ejecútela si una sesión murió sin terminar (sin GPU)

In [ ]:
# 🟢 CELDA 8 — ¿por qué terminó la sesión anterior? (OOM, corte externo o cierre ordenado)
from speakerscribe.batch import autopsy

print('\n'.join(autopsy(settings)))

## 🟡 9 · Apagado manual (el automático ya ocurre en la celda 6)

In [ ]:
# 🟡 CELDA 9 — liberar la máquina a mano (tras renombrar o revisar)
from speakerscribe.batch.colab import shutdown_runtime

APAGAR_AHORA = False  # ⇦ True + ejecutar = libera la GPU (cuenta regresiva cancelable)

if APAGAR_AHORA:
    shutdown_runtime(settings.shutdown_delay_s, 'manual')
else:
    print('ℹ️ Apagado manual desactivado.')

---
## 📖 Referencia operativa

| Escenario | Comportamiento |
|---|---|
| Sesión cortada por Colab | Nada confirmado se pierde; el archivo en curso se rehace. Los commits interrumpidos se recuperan solos. |
| Archivo nuevo | Entra solo: se copia al disco local mientras se calcula su huella (una sola lectura de Drive). |
| El mismo audio en dos rutas | Se transcribe una vez; el segundo reutiliza el resultado sin GPU (`reused`). |
| Activa un formato o cambia el encabezado | Se re-renderiza desde el JSON maestro, sin GPU (`republished`). |
| Renombró o editó un `.txt` (p. ej. quitó el `*` tras resumirlo) | Se respeta y se informa; nunca se sobrescribe. |
| Un archivo de `entregables/` que el lote no creó tiene el mismo nombre | El nuevo sale como `nombre~<id>.txt`; el suyo queda intacto. |
| Falla la diarización de UN archivo | Se reintenta en la próxima sesión (la caché de diarización evita repetirla). En el último intento se publica MARCADO. |
| Falla el entorno (librerías, CUDA, token) | `failed_environment`: no gasta intentos. Dos fallos idénticos seguidos detienen el lote y la máquina queda viva para leer el error. |
| RAM alta | Recicla modelos al 70 %; al 88 % cierra limpio. Reinicie la sesión y vuelva a ejecutar. |
| Dos sesiones a la vez | La segunda se niega (lock con latido). |
| Audio transcrito OK | Pasa a `_procesados/AAAA-MM-DD/` y se borra a los 30 días (todo queda en el journal). |

**Formato del `.txt`:**
```
# archivo_origen: 2026-10-01 *Reunión de seguimiento.mkv
# duracion: 01:02:07 · idioma: es (p=0.99)
# hablantes: 3 (SPEAKER_00, SPEAKER_01, SPEAKER_02)
[00:00:04 - 00:00:11] SPEAKER_00: Buenos días a todos…
```

> ⚠️ Ningún ASR ni diarizador es infalible: revise nombres propios y cifras antes de usar una transcripción como fuente.